In [4]:
# !pip install -U duckduckgo-search

### Step 2: Setup

In [5]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")
else:
    print("OPENAI_API_KEY is set.")


client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1-mini"

OPENAI_API_KEY is set.


### Step 3: Define the tools

In [6]:
def search_web(query: str):
    """Search the web using duckduckgo browser. Return 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"\u2705 Got results")
    return json.dumps(results, indent=2)

In [7]:
def fetch_url(url: str):
    """Fetching the content from url using trafilatura"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"\u2705 Got text: {len(text)} characters")
            return text
    print(f"\u274c Failed to extract text from {url}.")
    return f"Could not extract text from {url}. Try a diffrent source."

In [8]:
# Test search 
search_web("Apples in Madagascar")

✅ Got results


'[\n  {\n    "title": "Field Guide to Fruit on Madagascar - jurgenenkatja",\n    "href": "https://www.jurgenenkatja.nl/en/fruit-on-madagascar/",\n    "body": "English: Mimusops coriacea, Monkey\'s Apple, Mulch Apple Malagasy: Anganahara, Voranto, Vonanto Dutch: Mimusops coriacea It is also called the Malagasy apple. The outside is hard and you have to work to break the fruit open. This species originates from the Comoros and Madagascar. We came across this fruit on the east coast, but it also seems to grow, to a lesser extent, in the northwest. The ..."\n  },\n  {\n    "title": "Apple Blossoms in Madagascar: A Botanical Anomaly and Photographic ...",\n    "href": "https://frameandfocal.com/photography-glossary/apple-blossoms-in-madagascar",\n    "body": "Apple blossoms don\'t naturally grow in Madagascar\\u2014but small-scale orchards exist. This article details their origins, climate constraints, bloom timing (Sept-Oct), and precise photographic techniques for capturing them under hum

In [9]:
result = fetch_url("https://en.wikipedia.org/wiki/Annona_squamosa")
print(result)

✅ Got text: 15348 characters
Annona squamosa
| Annona squamosa |  | 
|---|---|
| Cross section of the fruit shown on right |  | 
| Scientific classification |  | 
| Kingdom: | Plantae | 
| Clade: | Tracheophytes | 
| Clade: | Angiosperms | 
| Clade: | Magnoliids | 
| Order: | Magnoliales | 
| Family: | Annonaceae | 
| Genus: | Annona | 
| Species: | A. squamosa | 
| Binomial name |  | 
| Annona squamosa |  | 
| Synonyms[3] |  | 
|  |  | 
Annona squamosa is a small, well-branched tree or shrub from the family Annonaceae that bears edible fruits called sugar apples, sweetsops or custard apples.[4][5][6] It tolerates a tropical lowland climate better than its relatives Annona reticulata and Annona cherimola,[7] and is the most widely cultivated of these Annona species.[5][8]
Annona squamosa is semi-(or late) deciduous,[9] and 3 to 8 metres (10 to 26 feet) tall,[4][9] similar to soursop (Annona muricata).[10] It is a native of tropical climate in the Americas, and Spanish traders aboard th

### Step 4: Describe as LLM tools

In [10]:
tools = []

In [11]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using duckduckgo browser. Returns 3 results",
    "parameters": {
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "The search query to find relevant websites"
            }
        },
        "required": ["query"]
    }
}

# Add pushover to the list of tools for LLM
tools.append({"type": "function", "function": search_web_function})

In [12]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch and extract the amin text content from a web page.",
    "parameters": {
        "type": "object",
        "properties":{
            "url":{
                "type": "string",
                "description": "The URL of the web page to fetch and extract text from."
            }
        },
        "required": ["url"]
    }
}

# Add pushover to the list of tools for LLM
tools.append({"type": "function", "function": fetch_url_function})

In [13]:
# tools

### Step 5: Tool call handler

In [14]:
def handle_tool_call(tool_calls):
    tool_result = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f" \U0001f527 Calling function {function_name}, with arguments{args}")
        
        # Route to the appropriate function based on the function name
        if function_name == "search_web":
            # Actually send the notification
            result = search_web(args["query"])
            content = f"Search result: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args["url"])
            content = f"Fetched URL content: {result}"
        
        else:
            content = f"Unknown funtion: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }

        tool_result.append(tool_call_result)

    return tool_result

In [15]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 3 distinct sources before delivering your brief. 
If you have fewer than 3 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imparetive that "DONE:" should be at the start of the final response, so that it can be easily parsed and extracted.
You CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINING of the final research brief

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step 6: Agentic loop 

In [16]:
def run_research_agent(topic: str, max_iterations: int=10) -> str:
    """
    Run research agent on a topic and return the research brief
    
    Args:
    topic - the topic to research
    max_iterations - safety limit to prevent infinite loops

    returns:
    The research brief as a string.
    """
    print(f"\n\U0001F50D Starting research on {topic}")
    print("=" *60)

    messages = [
        {"role": "system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and produce a comprehensive research brief: \n{topic}"}
    ]

    iteration = 0
    while iteration < max_iterations:
        iteration += 1 # just adds one to the iteration and the iteration starts with 1. 
        print(f"\n ---- Iteration {iteration} -----")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)

        else:
            content = message.content # No tool calls just a normal text from LLM
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"\n u2705 Research complete!")
                return research_brief
            else:
                print(f"  \U0001F4AD Agent is thinking: ")
                pprint(content)

        if (iteration == max_iterations - 1):
            print(f"   \u26a0 Safety limit reached. Stopping research in next iteration")
            messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please deliver your research brief now. You MUST respond with DONE: followed by your brief"})

    return "Research incomplete. Maximum iterations reached without finalizing research brief"


### Lets have it!


In [17]:
# brief = run_research_agent("Best PHD programs in Remote sensing in the United States")
# display(Markdown(brief))

### Evals

In [18]:
JUDGE_PROMPT ="""
# Judge Prompt: Insufficient Source Breadth (TRUE/FALSE)

You are scoring a research agent's **final deliverable research brief** to determine if there was a **source breadth failure**.
Return **only** `TRUE` or `FALSE`.

## Definitions

**Source:** A distinct, identifiable origin of information that the brief references. Examples: an article, report, paper, dataset, website page, book, filing, press release, interview, or named organization's publication. A source counts if the brief identifies it in any form: a URL, a title, an author or publisher name, a footnote, a bibliography entry, or an inline mention (e.g., "according to Gartner's 2025 CIO Survey…").

**Distinct source:** Each unique document or work counts once, no matter how many times the brief references it.
- Two different articles from the same publisher (e.g., two separate Reuters articles) count as **2** sources.
- The same document referenced several times, or reached through different URLs or formats, counts as **1** source.
- A source cited inside another source (e.g., "a McKinsey report cited by Forbes") counts only if the brief names it as its own identifiable work.

**What does NOT count as a source:**
- Vague or unattributed claims ("studies show," "experts say," "research suggests," "industry data indicates").
- The agent's own reasoning, general knowledge, or analysis.
- Search engines, tools, or platforms used to find information (e.g., "Google," "web search," "Perplexity"), unless a specific document hosted there is identified.
- The user's original request or instructions.
- Sources that appear only in the agent's trace or tool calls and never in the final brief.

**Out of scope:** Do **not** judge citation format, citation completeness, source quality, source credibility, or whether the source supports the claim. This judge measures **breadth only**.

**Source breadth failure (label TRUE):** Any of the below occurred:

1. **Fewer than 6 distinct sources:** The final brief references 5 or fewer distinct sources under the definitions above.
2. **Padding with duplicates:** The brief appears to list 6 or more sources, but after merging duplicates (same document, different URLs, formats, or repeated entries), fewer than 6 distinct sources remain.
3. **Padding with non-sources:** The brief reaches 6 or more only by counting items that do not qualify (vague attributions, tools, search engines, the user's prompt, the agent's own knowledge).
4. **Sources used but not in the deliverable:** The agent consulted 6 or more sources during research (visible in the trace), but the final brief references fewer than 6.
5. **No deliverable:** No final research brief was produced, or it is empty or truncated so badly that the sources can't be identified.

**No source breadth failure (label FALSE):** The final research brief references **6 or more distinct, identifiable sources**, counted under the definitions above. Formatting, placement (inline, footnotes, or bibliography), and citation style don't matter.

## Scoring Procedure

1. Read only the final deliverable brief. Use the trace only to confirm whether failure #4 or #5 applies.
2. List every candidate source referenced in the brief.
3. Remove items that don't qualify as sources.
4. Merge duplicates.
5. Count what remains. If the count is 6 or more, return `FALSE`. Otherwise, return `TRUE`.
6. If you can't tell whether an item is a distinct identifiable source, don't count it.

## Output Format

Return exactly one token: `TRUE` or `FALSE`. No explanations.
"""



In [19]:
JUDGE_MODEL = "gpt-4.1"

TOPICS = [
    "Impact of GLP-1 drugs on the US food and beverage industry",                 # health / business
    "State of solid-state battery commercialization for EVs",                     # tech / energy
    "Effects of remote work on commercial real estate valuations since 2020",     # economics / real estate
    "Causes and consequences of the collapse of the Bronze Age Mediterranean",    # history
    "EU AI Act: key obligations and compliance timeline for US companies",        # law / policy
    "Microplastics in drinking water: health risks and filtration methods",       # environment / health
    "Business model and growth of women's professional sports leagues",           # sports / business
    "Progress and challenges in nuclear fusion energy",                           # science
    "Rise of K-pop as a global soft-power strategy",                              # culture / geopolitics
    "Food insecurity among US college students",                                  # social policy / education
]


In [20]:
import contextlib
import io
import pandas as pd

def judge(brief: str) -> str:
    r = client.chat.completions.create(
        model=JUDGE_MODEL, temperature=0, max_tokens=3,
        messages=[{"role": "system", "content": JUDGE_PROMPT},
                  {"role": "user", "content": f"## Final Research Brief\n\n{brief or '[NO DELIVERABLE]'}"}],
    )
    out = r.choices[0].message.content.strip().strip("`").upper()
    return out if out in {"TRUE", "FALSE"} else f"INVALID: {out}"

rows = []
for topic in TOPICS:
    try:
        with contextlib.redirect_stdout(io.StringIO()): #suppress print output from agents for cleaner evaluation logs
            brief = run_research_agent(topic)
        rows.append({"topic": topic, "label": judge(brief), "brief": brief})
    except Exception as e:
        rows.append({"topic": topic, "label": f"ERROR: {e}", "brief": None})

df = pd.DataFrame(rows)
valid = df[df.label.isin(["TRUE", "FALSE"])]
print(f"Source-breadth failure rate: {(valid.label == 'TRUE').mean():.0%} "
      f"({(valid.label == 'TRUE').sum()}/{len(valid)} valid, {len(df) - len(valid)} errors/invalid)")
df[["topic", "label"]]

Source-breadth failure rate: 100% (5/5 valid, 5 errors/invalid)


,topic,label
0,Impact of GLP-1 drugs on the US food and bever...,TRUE
1,State of solid-state battery commercialization...,TRUE
2,Effects of remote work on commercial real esta...,TRUE
3,Causes and consequences of the collapse of the...,TRUE
4,EU AI Act: key obligations and compliance time...,TRUE
5,Microplastics in drinking water: health risks ...,ERROR: No results found.
6,Business model and growth of women's professio...,ERROR: No results found.
7,Progress and challenges in nuclear fusion energy,ERROR: No results found.
8,Rise of K-pop as a global soft-power strategy,ERROR: No results found.
9,Food insecurity among US college students,ERROR: No results found.
